# FR5 ACT Dataset — EDA

Explores the LeRobot v3.0 dataset produced by the SO-101 → FR5 teleoperation pipeline.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
from pathlib import Path

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

In [ ]:
DATASET_ROOT = Path("../../so101-fr5-teleop/lerobot_dataset")  # adjust if needed

with open(DATASET_ROOT / "meta/info.json") as f:
    info = json.load(f)

print(f"episodes : {info['total_episodes']}")
print(f"frames   : {info['total_frames']}")
print(f"fps      : {info['fps']}")
print(f"tasks    : {info['total_tasks']}")
print(f"version  : {info['codebase_version']}")

## Load data

In [ ]:
df = pq.read_table(DATASET_ROOT / "data/chunk-000/file-000.parquet").to_pandas()
episodes = pq.read_table(DATASET_ROOT / "meta/episodes/chunk-000/file-000.parquet").to_pandas()
tasks = pd.read_parquet(DATASET_ROOT / "meta/tasks.parquet")

print(f"data shape: {df.shape}")
df.head(3)

In [ ]:
print("tasks:")
print(tasks.to_string())

## Episode stats

In [ ]:
lengths = episodes["length"]
print(f"min length : {lengths.min()} frames  ({lengths.min()/info['fps']:.1f}s)")
print(f"max length : {lengths.max()} frames  ({lengths.max()/info['fps']:.1f}s)")
print(f"mean length: {lengths.mean():.0f} frames  ({lengths.mean()/info['fps']:.1f}s)")
print(f"std        : {lengths.std():.0f} frames")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(lengths, bins=20, color="#4C72B0", edgecolor="white", linewidth=0.5)
ax.axvline(lengths.mean(), color="#C44E52", linewidth=1.5, linestyle="--", label=f"mean={lengths.mean():.0f}")
ax.set_xlabel("frames per episode")
ax.set_ylabel("count")
ax.set_title("Episode length distribution")
ax.legend()
plt.tight_layout()
plt.show()

## Action space

In [ ]:
action = np.array(df["action"].tolist())   # (N, 7)  joints 1-6 + gripper
action_names = ["j1", "j2", "j3", "j4", "j5", "j6", "gripper"]

print("action stats (deg / norm):")
stats = pd.DataFrame({
    "min":  action.min(axis=0),
    "max":  action.max(axis=0),
    "mean": action.mean(axis=0),
    "std":  action.std(axis=0),
}, index=action_names)
print(stats.round(2).to_string())

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 5))
axes = axes.flatten()

for i, name in enumerate(action_names):
    axes[i].hist(action[:, i], bins=50, color="#4C72B0", edgecolor="none")
    axes[i].set_title(name)
    axes[i].set_ylabel("count" if i % 4 == 0 else "")

axes[-1].set_visible(False)
fig.suptitle("Action distributions", y=1.01)
plt.tight_layout()
plt.show()

## Observation state

In [ ]:
state = np.array(df["observation.state"].tolist())   # (N, 6)
eef   = np.array(df["observation.eef_pose"].tolist()) # (N, 6)

fig, axes = plt.subplots(2, 6, figsize=(16, 5))

for i in range(6):
    axes[0, i].hist(state[:, i], bins=50, color="#DD8452", edgecolor="none")
    axes[0, i].set_title(f"state j{i+1}")

eef_names = ["x_mm", "y_mm", "z_mm", "rx", "ry", "rz"]
for i in range(6):
    axes[1, i].hist(eef[:, i], bins=50, color="#55A868", edgecolor="none")
    axes[1, i].set_title(f"eef {eef_names[i]}")

plt.tight_layout()
plt.show()

## Action–state correlation

In [ ]:
corr = np.corrcoef(action[:, :6].T, state.T)[:6, 6:]

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
ax.set_xticks(range(6))
ax.set_xticklabels([f"state j{i+1}" for i in range(6)], rotation=30, ha="right")
ax.set_yticks(range(6))
ax.set_yticklabels([f"action j{i+1}" for i in range(6)])
ax.set_title("Pearson r — commanded vs actual joint positions")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Sample trajectories

In [ ]:
N_SHOW = min(5, info["total_episodes"])
colors = plt.cm.tab10.colors

fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=False)

for ep_idx in range(N_SHOW):
    ep = df[df["episode_index"] == ep_idx]
    t = ep["frame_index"].values
    a = np.array(ep["action"].tolist())
    s = np.array(ep["observation.state"].tolist())
    c = colors[ep_idx % len(colors)]

    axes[0].plot(t, a[:, 0], color=c, alpha=0.8, linewidth=0.9, label=f"ep {ep_idx}")
    axes[1].plot(t, a[:, 1], color=c, alpha=0.8, linewidth=0.9)
    axes[2].plot(t, a[:, 2], color=c, alpha=0.8, linewidth=0.9)
    axes[3].plot(t, a[:, 6], color=c, alpha=0.8, linewidth=0.9)

labels = ["cmd j1 (deg)", "cmd j2 (deg)", "cmd j3 (deg)", "gripper"]
for ax, lbl in zip(axes, labels):
    ax.set_ylabel(lbl)
    ax.set_xlabel("frame")

axes[0].legend(fontsize=7, ncol=N_SHOW)
fig.suptitle("Sample episode trajectories")
plt.tight_layout()
plt.show()

## Action smoothness (Δ per step)

In [ ]:
ep0 = df[df["episode_index"] == 0]
a0  = np.array(ep0["action"].tolist())
da  = np.diff(a0, axis=0)

fig, axes = plt.subplots(2, 1, figsize=(12, 5))

for i in range(6):
    axes[0].plot(da[:, i], alpha=0.7, linewidth=0.8, label=f"j{i+1}")
axes[1].plot(da[:, 6], color="#C44E52", linewidth=0.8)

axes[0].set_title("Action delta per step — joints (ep 0)")
axes[1].set_title("Action delta per step — gripper (ep 0)")
axes[0].set_ylabel("Δ deg")
axes[1].set_ylabel("Δ norm")
axes[0].legend(fontsize=7, ncol=6)
for ax in axes:
    ax.set_xlabel("step")
plt.tight_layout()
plt.show()

print(f"max |Δ| joints  : {np.abs(da[:, :6]).max():.3f} deg")
print(f"mean |Δ| joints : {np.abs(da[:, :6]).mean():.3f} deg")

## Data quality checks

In [ ]:
array_cols = ["observation.state", "action", "observation.eef_pose", "observation.joint_vel"]

print(f"{'column':<30} {'NaNs':>8} {'Infs':>8}")
print("-" * 50)
for col in array_cols:
    arr = np.array(df[col].tolist())
    nans = int(np.isnan(arr).sum())
    infs = int(np.isinf(arr).sum())
    flag = "  ← !!" if (nans + infs) > 0 else ""
    print(f"{col:<30} {nans:>8} {infs:>8}{flag}")

# check index continuity
assert df["index"].is_monotonic_increasing, "global index not monotonic"
print("\nglobal index: OK")

# check next.done appears exactly once per episode
done_per_ep = df.groupby("episode_index")["next.done"].sum()
bad = done_per_ep[done_per_ep != 1]
if len(bad) == 0:
    print("next.done   : OK (exactly 1 per episode)")
else:
    print(f"next.done   : WARN — {len(bad)} episodes have wrong done count")
    print(bad)

## Video frame sample

In [ ]:
import cv2

VIDEO_KEY = "observation.images.wrist_cam"
vid_dir   = DATASET_ROOT / "videos" / VIDEO_KEY / "chunk-000"
vid_files = sorted(vid_dir.glob("file-*.mp4")) if vid_dir.exists() else []

if not vid_files:
    print("no video files found")
else:
    n_show = min(4, len(vid_files))
    fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 3))
    if n_show == 1:
        axes = [axes]

    for ax, vf in zip(axes, vid_files[:n_show]):
        cap = cv2.VideoCapture(str(vf))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)
        ret, frame = cap.read()
        cap.release()
        if ret:
            ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(vf.stem, fontsize=8)
        ax.axis("off")

    fig.suptitle("Mid-episode frames (wrist cam)")
    plt.tight_layout()
    plt.show()